# Chapitre 10 - Exercices 10.1 à 10.6

Ce notebook traite les exercices 10.1 à 10.6 : le coefficient d'AdaBoost, le minimiseur de
la perte exponentielle, dépendance partielle et espérance conditionnelle, AdaBoost avec
des arbres (simulation (10.2) et classes qui se chevauchent), la perte exponentielle
multiclasse et le test de McNemar.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import optimize, stats
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
set_style()

from esl import datasets
from esl.ch10_boosting import (AdaBoostM1, GradientBoosting, RegressionTree,
                               simulate_nested_spheres, partial_dependence_bruteforce)

## Exercice 10.1 : Le coefficient $\beta_m$ d'AdaBoost

**Énoncé (reformulé).** Établir l'expression (10.12) du coefficient de mise à jour
d'AdaBoost : $\beta_m = \frac12\log\frac{1-\mathrm{err}_m}{\mathrm{err}_m}$.

**Rappels.** À l'étape $m$ de la modélisation pas à pas avec la perte exponentielle, on
cherche (éq. (10.9))

$$(\beta_m, G_m) = \arg\min_{\beta, G}\sum_{i=1}^N w_i^{(m)}\exp\big(-\beta\, y_i G(x_i)\big),
\qquad w_i^{(m)} = \exp\big(-y_i f_{m-1}(x_i)\big) > 0,$$

où $y_i \in\{-1,1\}$ et $G(x_i)\in\{-1,1\}$. L'erreur pondérée de $G$ est
$\mathrm{err} = \sum_i w_i^{(m)} I(y_i\ne G(x_i)) / \sum_i w_i^{(m)}$.

**Solution.**

*Étape 1 : réécriture.* Le produit $y_iG(x_i)$ vaut $+1$ si $x_i$ est bien classé et $-1$
sinon. Notons $W = \sum_i w_i^{(m)}$ et $W_{\text{err}} = \sum_i w_i^{(m)} I(y_i \ne G(x_i)) = W\,\mathrm{err}$. Alors

$$J(\beta) := \sum_i w_i^{(m)}e^{-\beta y_iG(x_i)} = e^{-\beta}\,(W - W_{\text{err}}) + e^{\beta}\,W_{\text{err}}
= W\big[e^{-\beta}(1-\mathrm{err}) + e^{\beta}\mathrm{err}\big].$$

*Étape 2 : choix de $G$.* On a aussi $J = (e^\beta - e^{-\beta})W_{\text{err}} + e^{-\beta}W$ (éq. (10.11)). Pour
tout $\beta > 0$, le coefficient $e^\beta - e^{-\beta}$ est positif : le meilleur $G$ est celui
qui minimise $W_{\text{err}}$, quel que soit $\beta>0$. On peut donc d'abord choisir $G_m$
(éq. (10.10)), puis optimiser $\beta$ avec $\mathrm{err} = \mathrm{err}_m$ fixé.

*Étape 3 : optimisation en $\beta$.* La fonction $h(\beta) = e^{-\beta}(1-\mathrm{err}_m) + e^{\beta}\mathrm{err}_m$
est dérivable sur $\mathbb R$ avec

$$h'(\beta) = -e^{-\beta}(1-\mathrm{err}_m) + e^{\beta}\mathrm{err}_m, \qquad h''(\beta) = e^{-\beta}(1-\mathrm{err}_m) + e^{\beta}\mathrm{err}_m > 0.$$

$h$ est donc strictement convexe (si $0<\mathrm{err}_m<1$) et son minimum est atteint à
l'unique zéro de $h'$ :

$$e^{\beta}\mathrm{err}_m = e^{-\beta}(1-\mathrm{err}_m) \iff e^{2\beta} = \frac{1-\mathrm{err}_m}{\mathrm{err}_m}
\iff \beta_m = \frac12\log\frac{1-\mathrm{err}_m}{\mathrm{err}_m}.$$

On a multiplié par $e^{\beta} > 0$ puis pris le logarithme (fonction strictement croissante).
Enfin $\beta_m > 0 \iff \mathrm{err}_m < 1/2$ : c'est cohérent avec l'hypothèse $\beta>0$ de l'étape 2,
dès que le classifieur faible fait mieux que le hasard.

> **Remarque.** Si $\mathrm{err}_m > 1/2$, on obtient $\beta_m < 0$ : on utilise alors $-G_m$,
> qui fait mieux que le hasard. Si $\mathrm{err}_m = 0$, $h$ est décroissante et $\beta_m = +\infty$
> (classifieur parfait) : en pratique on borne $\mathrm{err}_m$ loin de 0.

**Vérification numérique.** On prend une souche pondérée sur des données simulées avec
des poids quelconques, on minimise $J(\beta)$ numériquement et on compare à la formule.

In [ ]:
rng = np.random.RandomState(0)
X, y = simulate_nested_spheres(500, rng=rng)
w = rng.exponential(size=500)
G = np.where(RegressionTree(2).fit(X, y, w).predict(X) >= 0, 1, -1)
err = np.sum(w * (G != y)) / w.sum()
J = lambda b: np.sum(w * np.exp(-b * y * G))
b_num = optimize.minimize_scalar(J, bounds=(-5, 5), method="bounded", options={"xatol": 1e-10}).x
print("err_m = %.4f ; beta numérique = %.6f ; formule (10.12) = %.6f"
      % (err, b_num, 0.5 * np.log((1 - err) / err)))